# Cortex Cost Attribution Runbook

**Purpose**: SQL queries to analyze and attribute Cortex AI costs across users, models, warehouses, search services, semantic models, and agents.

**Last Updated**: March 2026

**Version**: 2.0

## 1. Configuration Variables

In [ ]:
-- 1.1 Configuration Variables
SET DATE_RANGE_DAYS = 30;  -- Number of days to look back
SET CREDIT_PRICE = 3.00;   -- Price per credit in USD (adjust based on your contract)

---
## 2. Overview: Total Cortex AI Costs

Get a high-level summary of all AI-related service charges.

In [ ]:
-- 2.1 Total AI_SERVICES credits from METERING_DAILY_HISTORY
SELECT 
    SUM(CREDITS_USED) AS total_credits,
    SUM(CREDITS_USED) * $CREDIT_PRICE AS estimated_cost_usd,
    MIN(USAGE_DATE) AS period_start,
    MAX(USAGE_DATE) AS period_end
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE());

In [ ]:
-- 2.2 Daily AI_SERVICES trend
SELECT 
    USAGE_DATE,
    CREDITS_USED AS credits,
    CREDITS_USED * $CREDIT_PRICE AS cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
ORDER BY USAGE_DATE DESC;

In [ ]:
-- 2.3 Credits breakdown by Cortex feature
SELECT 
    'Cortex AISQL (LLM Functions)' AS feature,
    SUM(TOKEN_CREDITS) AS credits,
    SUM(TOKENS) AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Cortex Search (Serving)' AS feature,
    SUM(CREDITS) AS credits,
    NULL AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_SERVING_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Cortex Analyst' AS feature,
    SUM(CREDITS) AS credits,
    NULL AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Cortex Fine-Tuning' AS feature,
    SUM(TOKEN_CREDITS) AS credits,
    SUM(TOKENS) AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FINE_TUNING_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Cortex Agents' AS feature,
    SUM(TOKEN_CREDITS) AS credits,
    SUM(TOKENS) AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Cortex Code CLI' AS feature,
    SUM(TOKEN_CREDITS) AS credits,
    SUM(TOKENS) AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Document Processing' AS feature,
    SUM(CREDITS_USED) AS credits,
    NULL AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

UNION ALL

SELECT 
    'Snowflake Intelligence' AS feature,
    SUM(TOKEN_CREDITS) AS credits,
    SUM(TOKENS) AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())

ORDER BY credits DESC NULLS LAST;

---
## 3. Cost Attribution by User

Identify which users are consuming Cortex resources.

In [ ]:
-- 3.1 Cortex AISQL usage by User
SELECT 
    u.NAME AS user_name,
    COUNT(DISTINCT a.QUERY_ID) AS query_count,
    SUM(a.TOKENS) AS total_tokens,
    SUM(a.TOKEN_CREDITS) AS total_credits,
    SUM(a.TOKEN_CREDITS) * $CREDIT_PRICE AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY a
JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON a.USER_ID = u.USER_ID
WHERE a.USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY u.NAME
ORDER BY total_credits DESC
LIMIT 20;

In [ ]:
-- 3.2 Cortex Analyst usage by User
SELECT 
    USERNAME AS user_name,
    SUM(REQUEST_COUNT) AS total_requests,
    SUM(CREDITS) AS total_credits,
    SUM(CREDITS) * $CREDIT_PRICE AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY USERNAME
ORDER BY total_credits DESC
LIMIT 20;

In [ ]:
-- 3.3 Cortex REST API usage by User
SELECT 
    u.NAME AS user_name,
    r.MODEL_NAME,
    SUM(r.TOKENS) AS total_tokens,
    r.INFERENCE_REGION
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_REST_API_USAGE_HISTORY r
JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON r.USER_ID = u.USER_ID
WHERE r.START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY u.NAME, r.MODEL_NAME, r.INFERENCE_REGION
ORDER BY total_tokens DESC
LIMIT 20;

---
## 4. Cost Attribution by Model

Identify which LLM models are driving costs.

In [ ]:
-- 4.1 Cortex AISQL credits by Model
SELECT 
    MODEL_NAME,
    FUNCTION_NAME,
    COUNT(DISTINCT QUERY_ID) AS query_count,
    SUM(TOKENS) AS total_tokens,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(TOKEN_CREDITS) * $CREDIT_PRICE AS estimated_cost_usd,
    ROUND(SUM(TOKEN_CREDITS) / NULLIF(SUM(TOKENS), 0) * 1000000, 4) AS credits_per_million_tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY MODEL_NAME, FUNCTION_NAME
ORDER BY total_credits DESC;

In [ ]:
-- 4.2 Daily model usage trend
SELECT 
    DATE_TRUNC('day', USAGE_TIME) AS usage_date,
    MODEL_NAME,
    SUM(TOKENS) AS total_tokens,
    SUM(TOKEN_CREDITS) AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY usage_date, MODEL_NAME
ORDER BY usage_date DESC, total_credits DESC;

---
## 5. Cost Attribution by Warehouse

Identify which warehouses are used for Cortex queries and their associated compute costs.

In [ ]:
-- 5.1 Cortex AISQL usage by Warehouse
SELECT 
    q.WAREHOUSE_NAME,
    COUNT(DISTINCT a.QUERY_ID) AS cortex_query_count,
    SUM(a.TOKEN_CREDITS) AS token_credits,
    SUM(q.CREDITS_ATTRIBUTED_COMPUTE) AS compute_credits,
    SUM(a.TOKEN_CREDITS) + SUM(q.CREDITS_ATTRIBUTED_COMPUTE) AS total_credits,
    (SUM(a.TOKEN_CREDITS) + SUM(q.CREDITS_ATTRIBUTED_COMPUTE)) * $CREDIT_PRICE AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY a
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY q ON a.QUERY_ID = q.QUERY_ID
WHERE a.USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY q.WAREHOUSE_NAME
ORDER BY total_credits DESC;

In [ ]:
-- 5.2 Individual query costs (Token + Compute)
SELECT 
    a.QUERY_ID, 
    q.WAREHOUSE_NAME,
    q.USER_NAME,
    q.START_TIME,
    a.MODEL_NAME,
    a.FUNCTION_NAME, 
    a.TOKENS,
    a.TOKEN_CREDITS, 
    q.CREDITS_ATTRIBUTED_COMPUTE,
    ZEROIFNULL(q.CREDITS_USED_QUERY_ACCELERATION) AS query_acceleration_credits,
    (a.TOKEN_CREDITS + q.CREDITS_ATTRIBUTED_COMPUTE + ZEROIFNULL(q.CREDITS_USED_QUERY_ACCELERATION)) AS total_credits,
    (a.TOKEN_CREDITS + q.CREDITS_ATTRIBUTED_COMPUTE + ZEROIFNULL(q.CREDITS_USED_QUERY_ACCELERATION)) * $CREDIT_PRICE AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY a
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY q ON a.QUERY_ID = q.QUERY_ID
WHERE a.USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
ORDER BY total_credits DESC
LIMIT 50;

---
## 6. Cost Attribution by Cortex Search Service

Detailed breakdown of Cortex Search Service costs.

In [ ]:
-- 6.1 Cortex Search Service hourly credits
SELECT 
    DATABASE_NAME,
    SCHEMA_NAME,
    SERVICE_NAME,
    SERVICE_ID,
    SUM(CREDITS) AS total_credits,
    SUM(CREDITS) * $CREDIT_PRICE AS estimated_cost_usd,
    MIN(START_TIME) AS first_usage,
    MAX(END_TIME) AS last_usage
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_SERVING_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY DATABASE_NAME, SCHEMA_NAME, SERVICE_NAME, SERVICE_ID
ORDER BY total_credits DESC;

In [ ]:
-- 6.2 Cortex Search Daily usage by Service and Consumption Type
SELECT 
    DATABASE_NAME,
    SCHEMA_NAME,
    SERVICE_NAME,
    CONSUMPTION_TYPE,
    MODEL_NAME,
    SUM(CREDITS) AS total_credits,
    SUM(CREDITS) * $CREDIT_PRICE AS estimated_cost_usd,
    SUM(TOKENS) AS total_tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
WHERE USAGE_DATE >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY DATABASE_NAME, SCHEMA_NAME, SERVICE_NAME, CONSUMPTION_TYPE, MODEL_NAME
ORDER BY total_credits DESC;

In [ ]:
-- 6.3 Daily trend per Cortex Search Service
SELECT 
    DATE_TRUNC('day', START_TIME) AS usage_date,
    DATABASE_NAME || '.' || SCHEMA_NAME || '.' || SERVICE_NAME AS service_fqn,
    SUM(CREDITS) AS daily_credits,
    SUM(CREDITS) * $CREDIT_PRICE AS daily_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_SERVING_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY usage_date, service_fqn
ORDER BY usage_date DESC, daily_credits DESC;

---
## 7. Cost Attribution by Semantic Model (Cortex Analyst)

Detailed breakdown of Cortex Analyst usage by semantic model/view.

**Fast Queries (7.1-7.3)**: Use `SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V` directly for request counts - no expensive joins required.

**Optional Compute Attribution (7.4)**: Joins with `QUERY_HISTORY` and `QUERY_ATTRIBUTION_HISTORY` using MD5 hash + timestamp matching for warehouse compute costs. This is slower but provides full cost attribution.

In [ ]:
-- 7.1 Semantic Model usage summary (FAST - no joins)
SELECT 
    SEMANTIC_MODEL_NAME,
    SEMANTIC_MODEL_TYPE,
    COUNT(*) AS total_requests,
    COUNT(CASE WHEN GENERATED_SQL IS NOT NULL THEN 1 END) AS successful_queries,
    COUNT(DISTINCT USER_NAME) AS unique_users,
    MIN(TIMESTAMP) AS first_request,
    MAX(TIMESTAMP) AS last_request
FROM SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V 
WHERE TIMESTAMP >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY SEMANTIC_MODEL_NAME, SEMANTIC_MODEL_TYPE
ORDER BY total_requests DESC;

In [ ]:
-- 7.2 Semantic Model usage by User (FAST - no joins)
SELECT 
    SEMANTIC_MODEL_NAME,
    USER_NAME,
    COUNT(*) AS total_requests,
    COUNT(CASE WHEN GENERATED_SQL IS NOT NULL THEN 1 END) AS successful_queries,
    COUNT(CASE WHEN RESPONSE_STATUS_CODE != 200 THEN 1 END) AS failed_requests
FROM SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V 
WHERE TIMESTAMP >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY SEMANTIC_MODEL_NAME, USER_NAME
ORDER BY total_requests DESC;

In [ ]:
-- 7.3 Daily trend by Semantic Model (FAST - no joins)
SELECT 
    DATE_TRUNC('day', TIMESTAMP) AS usage_date,
    SEMANTIC_MODEL_NAME,
    COUNT(*) AS total_requests,
    COUNT(CASE WHEN GENERATED_SQL IS NOT NULL THEN 1 END) AS successful_queries
FROM SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V 
WHERE TIMESTAMP >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY usage_date, SEMANTIC_MODEL_NAME
ORDER BY usage_date DESC, total_requests DESC;

In [ ]:
-- 7.4 Compute cost attribution by Semantic Model (OPTIONAL - slower, uses MD5 hash join)
-- Use this query when you need warehouse compute costs attributed to semantic models
WITH analyst_requests AS (
    SELECT 
        TIMESTAMP AS request_time,
        SEMANTIC_MODEL_NAME,
        USER_NAME,
        MD5(REPLACE(REPLACE(GENERATED_SQL, ' ', ''), CHR(10), '')) AS sql_hash
    FROM SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V 
    WHERE TIMESTAMP >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
      AND GENERATED_SQL IS NOT NULL
),
analyst_queries AS (
    SELECT 
        qh.QUERY_ID,
        qh.USER_NAME,
        qh.START_TIME,
        MD5(REPLACE(REPLACE(qh.QUERY_TEXT, ' ', ''), CHR(10), '')) AS sql_hash
    FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh
    WHERE qh.QUERY_TEXT LIKE '%-- Generated by Cortex Analyst%'
      AND qh.START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
)
SELECT 
    ar.SEMANTIC_MODEL_NAME,
    COUNT(DISTINCT aq.QUERY_ID) AS query_count,
    SUM(COALESCE(qa.CREDITS_ATTRIBUTED_COMPUTE, 0)) AS compute_credits,
    SUM(COALESCE(qa.CREDITS_ATTRIBUTED_COMPUTE, 0)) * $CREDIT_PRICE AS estimated_compute_cost_usd
FROM analyst_requests ar
JOIN analyst_queries aq 
    ON ar.sql_hash = aq.sql_hash
    AND ar.USER_NAME = aq.USER_NAME
    AND aq.START_TIME BETWEEN ar.request_time AND DATEADD(minute, 5, ar.request_time)
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY qa
    ON aq.QUERY_ID = qa.QUERY_ID
GROUP BY ar.SEMANTIC_MODEL_NAME
ORDER BY compute_credits DESC;

---
## 7b. Cortex Agents Cost Attribution

In [ ]:
-- 7b.1 Cortex Agent usage summary
SELECT 
    AGENT_NAME,
    USER_NAME,
    COUNT(*) AS total_requests,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(TOKEN_CREDITS) * $CREDIT_PRICE AS estimated_cost_usd,
    SUM(TOKENS) AS total_tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY AGENT_NAME, USER_NAME
ORDER BY total_credits DESC;

---
## 7c. Cortex Provisioned Throughput

For accounts with provisioned throughput (PTU) commitments — fixed-capacity model allocations.

In [ ]:
-- 7c.1 Provisioned Throughput usage and credits
SELECT 
    PROVISIONED_THROUGHPUT_ID,
    AI_SERVICE,
    MODEL_NAME,
    CLOUD_SERVICE_PROVIDER,
    TERM_START_DATE,
    TERM_END_DATE,
    SUM(PTU_COUNT) AS total_ptus,
    SUM(PTU_CREDITS) AS total_ptu_credits,
    SUM(PTU_CREDITS) * $CREDIT_PRICE AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY
WHERE INTERVAL_START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY PROVISIONED_THROUGHPUT_ID, AI_SERVICE, MODEL_NAME, CLOUD_SERVICE_PROVIDER, TERM_START_DATE, TERM_END_DATE
ORDER BY total_ptu_credits DESC;

---
## 7d. Cortex Code (CoCo) CLI Usage

Usage from Cortex Code CLI sessions — token credits and tokens per user.

In [ ]:
-- 7d.1 Cortex Code CLI usage by User
SELECT 
    USER_ID,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(TOKEN_CREDITS) * $CREDIT_PRICE AS estimated_cost_usd,
    SUM(TOKENS) AS total_tokens,
    COUNT(*) AS total_requests,
    MIN(USAGE_TIME) AS first_usage,
    MAX(USAGE_TIME) AS last_usage
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY USER_ID
ORDER BY total_credits DESC;

---
## 7e. Document Processing Usage

Cortex document processing (PARSE_DOCUMENT, etc.) and legacy Document AI usage.

In [ ]:
-- 7e.1 Cortex Document Processing usage by function and model
SELECT 
    FUNCTION_NAME,
    MODEL_NAME,
    OPERATION_NAME,
    COUNT(DISTINCT QUERY_ID) AS query_count,
    SUM(CREDITS_USED) AS total_credits,
    SUM(CREDITS_USED) * $CREDIT_PRICE AS estimated_cost_usd,
    SUM(PAGE_COUNT) AS total_pages,
    SUM(DOCUMENT_COUNT) AS total_documents
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY FUNCTION_NAME, MODEL_NAME, OPERATION_NAME
ORDER BY total_credits DESC;

In [ ]:
-- 7e.2 Legacy Document AI usage by operation
SELECT 
    OPERATION_NAME,
    COUNT(DISTINCT QUERY_ID) AS query_count,
    SUM(CREDITS_USED) AS total_credits,
    SUM(CREDITS_USED) * $CREDIT_PRICE AS estimated_cost_usd,
    SUM(PAGE_COUNT) AS total_pages,
    SUM(DOCUMENT_COUNT) AS total_documents
FROM SNOWFLAKE.ACCOUNT_USAGE.DOCUMENT_AI_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY OPERATION_NAME
ORDER BY total_credits DESC;

---
## 7f. Snowflake Intelligence Usage

Snowflake Intelligence (natural language querying) usage by intelligence object and user.

In [ ]:
-- 7f.1 Snowflake Intelligence usage by object and user
SELECT 
    SNOWFLAKE_INTELLIGENCE_NAME,
    USER_NAME,
    COUNT(*) AS total_requests,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(TOKEN_CREDITS) * $CREDIT_PRICE AS estimated_cost_usd,
    SUM(TOKENS) AS total_tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
GROUP BY SNOWFLAKE_INTELLIGENCE_NAME, USER_NAME
ORDER BY total_credits DESC;

---
## 8. Advanced: Combined Cost Analysis

In [ ]:
-- 8.1 Unified Cortex cost view across all services
WITH cortex_costs AS (
    SELECT 
        'Cortex AISQL' AS service,
        DATE_TRUNC('day', USAGE_TIME) AS usage_date,
        MODEL_NAME,
        SUM(TOKEN_CREDITS) AS credits,
        SUM(TOKENS) AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date, MODEL_NAME
    
    UNION ALL
    
    SELECT 
        'Cortex Search' AS service,
        DATE_TRUNC('day', START_TIME) AS usage_date,
        SERVICE_NAME AS MODEL_NAME,
        SUM(CREDITS) AS credits,
        NULL AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_SERVING_USAGE_HISTORY
    WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date, SERVICE_NAME
    
    UNION ALL
    
    SELECT 
        'Cortex Analyst' AS service,
        DATE_TRUNC('day', START_TIME) AS usage_date,
        'Analyst' AS MODEL_NAME,
        SUM(CREDITS) AS credits,
        NULL AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
    WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date
    
    UNION ALL
    
    SELECT 
        'Cortex Agents' AS service,
        DATE_TRUNC('day', START_TIME) AS usage_date,
        AGENT_NAME AS MODEL_NAME,
        SUM(TOKEN_CREDITS) AS credits,
        SUM(TOKENS) AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
    WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date, AGENT_NAME
    
    UNION ALL
    
    SELECT 
        'Cortex Code CLI' AS service,
        DATE_TRUNC('day', USAGE_TIME)::TIMESTAMP_LTZ AS usage_date,
        'CoCo CLI' AS MODEL_NAME,
        SUM(TOKEN_CREDITS) AS credits,
        SUM(TOKENS) AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date
    
    UNION ALL
    
    SELECT 
        'Document Processing' AS service,
        DATE_TRUNC('day', START_TIME) AS usage_date,
        COALESCE(MODEL_NAME, OPERATION_NAME) AS MODEL_NAME,
        SUM(CREDITS_USED) AS credits,
        NULL AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY
    WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date, COALESCE(MODEL_NAME, OPERATION_NAME)
    
    UNION ALL
    
    SELECT 
        'Snowflake Intelligence' AS service,
        DATE_TRUNC('day', START_TIME) AS usage_date,
        SNOWFLAKE_INTELLIGENCE_NAME AS MODEL_NAME,
        SUM(TOKEN_CREDITS) AS credits,
        SUM(TOKENS) AS tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY
    WHERE START_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
    GROUP BY usage_date, SNOWFLAKE_INTELLIGENCE_NAME
)
SELECT 
    service,
    usage_date,
    MODEL_NAME,
    credits,
    credits * $CREDIT_PRICE AS cost_usd,
    tokens
FROM cortex_costs
ORDER BY usage_date DESC, credits DESC;

In [ ]:
-- 8.2 Top 10 most expensive Cortex queries
SELECT 
    a.QUERY_ID,
    q.USER_NAME,
    q.WAREHOUSE_NAME,
    a.MODEL_NAME,
    a.FUNCTION_NAME,
    a.TOKENS,
    a.TOKEN_CREDITS AS token_cost,
    q.CREDITS_ATTRIBUTED_COMPUTE AS compute_cost,
    (a.TOKEN_CREDITS + q.CREDITS_ATTRIBUTED_COMPUTE) AS total_credits,
    (a.TOKEN_CREDITS + q.CREDITS_ATTRIBUTED_COMPUTE) * $CREDIT_PRICE AS total_cost_usd,
    q.START_TIME
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY a
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY q ON a.QUERY_ID = q.QUERY_ID
WHERE a.USAGE_TIME >= DATEADD(day, -$DATE_RANGE_DAYS, CURRENT_DATE())
ORDER BY total_credits DESC
LIMIT 10;

In [ ]:
-- 8.3 Week-over-week cost comparison
WITH weekly_costs AS (
    SELECT 
        DATE_TRUNC('week', USAGE_DATE) AS week_start,
        SUM(CREDITS_USED) AS weekly_credits
    FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
    WHERE SERVICE_TYPE = 'AI_SERVICES'
      AND USAGE_DATE >= DATEADD(day, -60, CURRENT_DATE())
    GROUP BY week_start
)
SELECT 
    week_start,
    weekly_credits,
    weekly_credits * $CREDIT_PRICE AS weekly_cost_usd,
    LAG(weekly_credits) OVER (ORDER BY week_start) AS prev_week_credits,
    ROUND((weekly_credits - LAG(weekly_credits) OVER (ORDER BY week_start)) / 
          NULLIF(LAG(weekly_credits) OVER (ORDER BY week_start), 0) * 100, 2) AS wow_change_pct
FROM weekly_costs
ORDER BY week_start DESC;

---
## 9. Summary & Recommendations

### Key Views Used
| View | Purpose |
|------|---------|
| `METERING_DAILY_HISTORY` | High-level AI_SERVICES credit summary |
| `CORTEX_AISQL_USAGE_HISTORY` | Per-query LLM function usage with user/warehouse |
| `CORTEX_SEARCH_SERVING_USAGE_HISTORY` | Hourly Cortex Search serving costs |
| `CORTEX_SEARCH_DAILY_USAGE_HISTORY` | Daily Cortex Search costs with consumption type |
| `CORTEX_ANALYST_USAGE_HISTORY` | Cortex Analyst usage by user |
| `CORTEX_AGENT_USAGE_HISTORY` | Cortex Agent usage by agent and user |
| `QUERY_ATTRIBUTION_HISTORY` | Compute costs per query |
| `CORTEX_CODE_CLI_USAGE_HISTORY` | Cortex Code CLI usage by user |
| `CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY` | Document processing (PARSE_DOCUMENT) costs |
| `DOCUMENT_AI_USAGE_HISTORY` | Legacy Document AI costs |
| `CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY` | Provisioned throughput (PTU) commitments |
| `SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY` | Snowflake Intelligence usage |

---
## Appendix: Data Latency Notes

| View | Latency |
|------|---------|
| `METERING_DAILY_HISTORY` | Up to 3 hours |
| `CORTEX_AISQL_USAGE_HISTORY` | Up to 3 hours |
| `CORTEX_SEARCH_*` | Up to 3 hours |
| `CORTEX_ANALYST_USAGE_HISTORY` | Up to 3 hours |
| `CORTEX_AGENT_USAGE_HISTORY` | Up to 3 hours |
| `QUERY_ATTRIBUTION_HISTORY` | Up to 3 hours |
| `CORTEX_CODE_CLI_USAGE_HISTORY` | Up to 3 hours |
| `CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY` | Up to 3 hours |
| `DOCUMENT_AI_USAGE_HISTORY` | Up to 3 hours |
| `CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY` | Up to 3 hours |
| `SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY` | Up to 3 hours |

---

**Version:** 2.1  
**Last Updated:** March 2026